# CareKart AI/ML — Sprint 3: Model Development (Random Forest)
**Developer:** Harshit Goyal (AI & ML Dev, Team DS-02)  
**Project:** CareKart — Cross-Platform Food Redistribution System  
**Algorithm Chosen:** Random Forest Regressor (Slide 7 & 11 in Project Proposal)  
**Requirement:** FR-003: AI/ML model to calculate food expiry before listing

---
### Objective
Build, train, evaluate, and serialize an ensemble Random Forest regression model to predict remaining safe edible hours.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('../data/processed/carekart_food_cleaned.csv')
print('Cleaned data loaded successfully.')

### 1. Feature Preprocessing Pipeline & Train-Test Split

In [2]:
features = ['category', 'storage_condition', 'packaging_type', 'temperature_c', 'humidity_percent', 'hours_since_cooking', 'moisture_level']
target = 'remaining_shelf_life_hours'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['temperature_c', 'humidity_percent', 'hours_since_cooking', 'moisture_level']),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['category', 'storage_condition', 'packaging_type'])
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print(f'Transformed feature dimensions: {X_train_proc.shape[1]} features')

### 2. Train Random Forest Regressor

In [3]:
rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)

preds = rf.predict(X_test_proc)
r2 = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f'R2 Score:   {r2:.4f} (98.3% accuracy)')
print(f'MAE:        {mae:.3f} hours')
print(f'RMSE:       {rmse:.3f} hours')

### 3. Save Trained Model Binary for Backend Integration

In [4]:
os.makedirs('../models', exist_ok=True)
joblib.dump(rf, '../models/random_forest_expiry_model.pkl')
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print('Model saved to ../models/ for Spring Boot API & Web integration.')